# SAXSFormer — Phase 3 Training

**What this notebook does:**
1. Upload `saxs_dataset_prepared.npz`
2. Verify GPU
3. Train SAXSFormer Transformer
4. Evaluate on test set → MAE, RMSE, R² per target
5. Save checkpoint + plots
6. Download all outputs

**You only need one file:** `saxs_dataset_prepared.npz`

## Cell 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > GPU')

## Cell 2 — Upload Dataset
Run this cell and upload `saxs_dataset_prepared.npz` when prompted.

In [ ]:
from google.colab import files
import os

os.makedirs('data/processed', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('reports/figures', exist_ok=True)

uploaded = files.upload()

# Move uploaded file to expected path
for filename in uploaded:
    os.rename(filename, f'data/processed/{filename}')
    print(f'Saved → data/processed/{filename}')

# Verify
path = 'data/processed/saxs_dataset_prepared.npz'
if os.path.exists(path):
    size_mb = os.path.getsize(path) / 1e6
    print(f'File ready: {path}  ({size_mb:.2f} MB)')
else:
    print('ERROR: File not found at expected path.')

## Cell 3 — Verify Dataset

In [ ]:
import numpy as np

with np.load('data/processed/saxs_dataset_prepared.npz', allow_pickle=True) as d:
    print('Keys       :', list(d.keys()))
    print('x shape    :', d['x'].shape)
    print('q shape    :', d['q'].shape)
    print('y_rg shape :', d['y_rg'].shape)
    print('y_dmax     :', d['y_dmax'].shape)
    print('y_volume   :', d['y_volume'].shape)
    print('NaN in x   :', np.isnan(d['x']).sum())
    print('NaN in y_rg:', np.isnan(d['y_rg']).sum())
    print('All good!  Ready for training.')

## Cell 4 — Imports & Config

In [ ]:
import math
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_PATH   = 'data/processed/saxs_dataset_prepared.npz'
CHECKPOINT_DIR = Path('checkpoints')
FIGURES_DIR    = Path('reports/figures')
SCALER_PATH    = Path('checkpoints/label_scaler.pkl')

# Model
D_MODEL   = 64
N_HEADS   = 4
N_LAYERS  = 4
D_FF      = 256
DROPOUT   = 0.1

# Training
BATCH_SIZE   = 32
EPOCHS       = 150
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 20

RANDOM_SEED  = 42
TARGET_NAMES = ['Rg (Å)', 'Dmax (Å)', 'Volume (Å³)']

# ── Seed + Device ─────────────────────────────────────────────────────────────
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

print('Config loaded.')

## Cell 5 — Load Data & Split

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
with np.load(DATASET_PATH, allow_pickle=True) as data:
    X        = data['x'].astype(np.float32)
    y_rg     = data['y_rg'].astype(np.float32)
    y_dmax   = data['y_dmax'].astype(np.float32)
    y_volume = data['y_volume'].astype(np.float32)

N, M = X.shape
Y    = np.stack([y_rg, y_dmax, y_volume], axis=1).astype(np.float32)

print(f'Samples       : {N}')
print(f'Curve length  : {M}')
print(f'Rg    → min={y_rg.min():.2f}  max={y_rg.max():.2f}  mean={y_rg.mean():.2f}')
print(f'Dmax  → min={y_dmax.min():.2f}  max={y_dmax.max():.2f}  mean={y_dmax.mean():.2f}')
print(f'Vol   → min={y_volume.min():.0f}  max={y_volume.max():.0f}  mean={y_volume.mean():.0f}')

# ── 70 / 15 / 15 Split ────────────────────────────────────────────────────────
X_tr, X_tmp, Y_tr, Y_tmp = train_test_split(X, Y, test_size=0.30, random_state=RANDOM_SEED)
X_val, X_te, Y_val, Y_te = train_test_split(X_tmp, Y_tmp, test_size=0.50, random_state=RANDOM_SEED)

print(f'\nSplit → train={len(X_tr)}  val={len(X_val)}  test={len(X_te)}')

# ── Label scaling (fit on train only) ─────────────────────────────────────────
label_scaler = StandardScaler()
Y_tr_n  = label_scaler.fit_transform(Y_tr).astype(np.float32)
Y_val_n = label_scaler.transform(Y_val).astype(np.float32)
Y_te_n  = label_scaler.transform(Y_te).astype(np.float32)
Y_te_raw = Y_te.copy()   # keep unscaled for evaluation

# Save scaler
with open(SCALER_PATH, 'wb') as f:
    pickle.dump(label_scaler, f)
print(f'Scaler saved → {SCALER_PATH}')

# ── DataLoaders ───────────────────────────────────────────────────────────────
def make_loader(X, Y, shuffle):
    ds = TensorDataset(
        torch.from_numpy(X).unsqueeze(-1),  # (N, 128, 1)
        torch.from_numpy(Y)
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, pin_memory=True)

train_loader = make_loader(X_tr,  Y_tr_n,  shuffle=True)
val_loader   = make_loader(X_val, Y_val_n, shuffle=False)
test_loader  = make_loader(X_te,  Y_te_n,  shuffle=False)

seq_len = M
print('DataLoaders ready.')

## Cell 6 — Define SAXSFormer Model

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding.
    Gives the Transformer awareness of each q-point's position
    without learned parameters — better at small data scale.
    """
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe           = torch.zeros(max_len, d_model)
        position     = torch.arange(max_len).unsqueeze(1).float()
        div_term     = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class SAXSFormer(nn.Module):
    """
    Transformer encoder → global average pool → MLP head.

    Input  : (batch, seq_len=128, 1)   one log I(q) value per token
    Output : (batch, 3)                [Rg, Dmax, Volume] normalised
    """
    def __init__(self, seq_len, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT, n_outputs=3):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len=seq_len+10, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True,
            norm_first=True,   # Pre-LN: more stable gradients
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_outputs),
        )

    def forward(self, x):
        x = self.input_proj(x)   # (batch, seq_len, d_model)
        x = self.pos_enc(x)
        x = self.encoder(x)      # (batch, seq_len, d_model)
        x = x.mean(dim=1)        # global average pool
        return self.head(x)      # (batch, 3)


model    = SAXSFormer(seq_len=seq_len).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'SAXSFormer ready  |  trainable parameters: {n_params:,}')
print(model)

## Cell 7 — Train

In [ ]:
criterion  = nn.MSELoss()
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

ckpt_path     = CHECKPOINT_DIR / 'saxsformer_best.pt'
best_val      = float('inf')
patience_ctr  = 0
train_losses  = []
val_losses    = []

print(f'Training for up to {EPOCHS} epochs  |  early stop patience={PATIENCE}\n')

for epoch in range(1, EPOCHS + 1):

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    for X_b, Y_b in train_loader:
        X_b, Y_b = X_b.to(device, non_blocking=True), Y_b.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(X_b), Y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(X_b)
    train_loss = epoch_loss / len(train_loader.dataset)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, Y_b in val_loader:
            X_b, Y_b = X_b.to(device, non_blocking=True), Y_b.to(device, non_blocking=True)
            val_loss += criterion(model(X_b), Y_b).item() * len(X_b)
    val_loss /= len(val_loader.dataset)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f'  Epoch {epoch:4d}/{EPOCHS}  '
              f'train={train_loss:.5f}  '
              f'val={val_loss:.5f}  '
              f'lr={scheduler.get_last_lr()[0]:.2e}')

    # ── Checkpoint + early stopping ──────────────────────────────────────────
    if val_loss < best_val:
        best_val     = val_loss
        patience_ctr = 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\n  [early stop] Stopped at epoch {epoch}')
            break

print(f'\nBest val loss = {best_val:.6f}')
print(f'Checkpoint saved → {ckpt_path}')

# Reload best weights
model.load_state_dict(torch.load(ckpt_path, map_location=device))
print('Best weights loaded.')

## Cell 8 — Evaluate on Test Set

In [ ]:
model.eval()
all_preds = []

with torch.no_grad():
    for X_b, _ in test_loader:
        all_preds.append(model(X_b.to(device, non_blocking=True)).cpu().numpy())

preds_raw = label_scaler.inverse_transform(np.vstack(all_preds))

metrics = {
    'MAE':  [mean_absolute_error(Y_te_raw[:, i], preds_raw[:, i]) for i in range(3)],
    'RMSE': [math.sqrt(mean_squared_error(Y_te_raw[:, i], preds_raw[:, i])) for i in range(3)],
    'R2':   [r2_score(Y_te_raw[:, i], preds_raw[:, i]) for i in range(3)],
}

print('=' * 55)
print('  SAXSFormer — Test Set Results')
print('=' * 55)
print(f"  {'Target':<14} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print('-' * 55)
for i, name in enumerate(TARGET_NAMES):
    print(f"  {name:<14}  {metrics['MAE'][i]:>8.3f}  {metrics['RMSE'][i]:>8.3f}  {metrics['R2'][i]:>8.3f}")
print('=' * 55)

## Cell 9 — Plot Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
epochs  = range(1, len(train_losses) + 1)

ax.plot(epochs, train_losses, label='Train loss',      linewidth=2)
ax.plot(epochs, val_losses,   label='Validation loss', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss (normalised labels)', fontsize=12)
ax.set_title('SAXSFormer — Training & Validation Loss', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

fig.tight_layout()
path = FIGURES_DIR / 'loss_curves.png'
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {path}')

## Cell 10 — Plot Predicted vs True

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors    = ['#1f77b4', '#2ca02c', '#d62728']

for i, (ax, name, color) in enumerate(zip(axes, TARGET_NAMES, colors)):
    y_t = Y_te_raw[:, i]
    y_p = preds_raw[:, i]

    ax.scatter(y_t, y_p, s=25, alpha=0.65, color=color, edgecolors='none')

    lo, hi = min(y_t.min(), y_p.min()), max(y_t.max(), y_p.max())
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.2, label='Perfect prediction')

    ax.set_xlabel(f'True {name}',      fontsize=11)
    ax.set_ylabel(f'Predicted {name}', fontsize=11)
    ax.set_title(
        f'{name}\n'
        f"R²={metrics['R2'][i]:.3f}  "
        f"MAE={metrics['MAE'][i]:.3f}  "
        f"RMSE={metrics['RMSE'][i]:.3f}",
        fontsize=11
    )
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('SAXSFormer — Predicted vs True (Test Set)', fontsize=14, y=1.02)
fig.tight_layout()
path = FIGURES_DIR / 'pred_vs_true.png'
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {path}')

## Cell 11 — Download All Outputs
Downloads the checkpoint, scaler, and both figures to your local machine.

In [ ]:
from google.colab import files

outputs = [
    'checkpoints/saxsformer_best.pt',
    'checkpoints/label_scaler.pkl',
    'reports/figures/loss_curves.png',
    'reports/figures/pred_vs_true.png',
]

for path in outputs:
    if os.path.exists(path):
        files.download(path)
        print(f'Downloaded: {path}')
    else:
        print(f'NOT FOUND:  {path}')